<a href="https://colab.research.google.com/github/Terenceisstudying/SIT-UofG-QC-Assignment/blob/main/BB84-Plain.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
%pip install qiskit==1.2.4
%pip install qiskit-aer==0.15.1
%pip install pylatexenc==2.10

from qiskit import QuantumCircuit
from qiskit.converters import circuit_to_gate
from qiskit.visualization import array_to_latex
from qiskit.quantum_info import Operator
from qiskit.quantum_info import Statevector
from qiskit import transpile
from qiskit.providers.basic_provider import BasicSimulator
from qiskit.visualization import plot_histogram
from qiskit.circuit import ControlledGate
import math

In [4]:
# The aim of the assignment is to simulate the BB84 key distribution protocol.

# This notebook is for a simulation of the protocol without an attacker.


# Using the Alice and Bob example.
# Alice creates random bits and random basis.
# Alice encodes each bit as a qubit.
# Alice send qubits to Bob.
# Bob chooses random basis and measures.
# Alice and Bob compare basis.
# Bob send bases to Alice
# Alice send only matching positions basis to Bob.



In [5]:
# Alice creates random Quantum bit
# To generate genuinely random numbers, we need a physical process that is random.
# For example: construct |+> and measure it.

def random_quantum_bit():
    qc = QuantumCircuit(1,1)
    qc.h(0)
    qc.measure(0,0)

    backend = BasicSimulator()
    compiled_circuit = transpile(qc, backend)
    job = backend.run(compiled_circuit, shots=1)
    result = job.result()
    counts = result.get_counts()

    return int(list(counts.keys())[0])

# List of qubits
def random_quantum_bits(n):
    return [random_quantum_bit() for _ in range(n)]

In [6]:
# Alice encodes each bit
# bit|basis   |state
# 0  |standard| |0>
# 1  |standard| |1>
# 0  |diagonal| |+>
# 1  |diagonal| |->
def alice_encode_qubit(bit, basis):
    qc = QuantumCircuit(1, 1)
    if bit == 1:
        qc.x(0)

    if basis == 1:
        qc.h(0)

    return qc

In [7]:
# Bob measures qubit

def bob_measure(prepared_qubit, basis):
    qc = prepared_qubit.copy()

    # if measuring in in diagonal basis (1), apply H
    if basis == 1:
        qc.h(0)
    # if measuring in standard basis (0), just do it
    qc.measure(0, 0)

    backend = BasicSimulator()
    compiled = transpile(qc, backend)
    job = backend.run(compiled, shots=1)
    result = job.result()
    counts = result.get_counts(compiled)

    return int(list(counts.keys())[0])

In [10]:
n = 19

# Alice
alice_bits = random_quantum_bits(n)
alice_basis = random_quantum_bits(n)

qubits = []
for i in range(n):
    qubits.append(alice_encode_qubit(alice_bits[i], alice_basis[i]))

# Bob
bob_basis = random_quantum_bits(n)

bob_bits = []
for i in range(n):
    bob_bits.append(bob_measure(qubits[i], bob_basis[i]))

# Public basis comparison
matching_positions = []
for i in range(n):
    if alice_basis[i] == bob_basis[i]:
        matching_positions.append(i)
    alice_key = [alice_bits[i] for i in matching_positions]
    bob_key = [bob_bits[i] for i in matching_positions]

    print("Alice bits: ", alice_bits)
    print("Alice bases:", alice_basis)
    print("Bob bases:  ", bob_basis)
    print("Bob bits:   ", bob_bits)
    print("Matching positions:", matching_positions)
    print("Alice key:", alice_key)
    print("Bob key:  ", bob_key)
    print("Keys match:", alice_key == bob_key)

Alice bits:  [1, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 0, 1, 1, 1]
Alice bases: [0, 0, 0, 0, 1, 1, 0, 0, 0, 1, 1, 1, 0, 0, 1, 0, 0, 0, 0]
Bob bases:   [0, 0, 1, 0, 0, 0, 0, 1, 1, 0, 1, 1, 1, 0, 0, 1, 0, 0, 0]
Bob bits:    [1, 0, 1, 1, 0, 1, 1, 1, 1, 0, 1, 1, 0, 1, 1, 1, 1, 1, 1]
Matching positions: [0]
Alice key: [1]
Bob key:   [1]
Keys match: True
Alice bits:  [1, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 0, 1, 1, 1]
Alice bases: [0, 0, 0, 0, 1, 1, 0, 0, 0, 1, 1, 1, 0, 0, 1, 0, 0, 0, 0]
Bob bases:   [0, 0, 1, 0, 0, 0, 0, 1, 1, 0, 1, 1, 1, 0, 0, 1, 0, 0, 0]
Bob bits:    [1, 0, 1, 1, 0, 1, 1, 1, 1, 0, 1, 1, 0, 1, 1, 1, 1, 1, 1]
Matching positions: [0, 1]
Alice key: [1, 0]
Bob key:   [1, 0]
Keys match: True
Alice bits:  [1, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 0, 1, 1, 1]
Alice bases: [0, 0, 0, 0, 1, 1, 0, 0, 0, 1, 1, 1, 0, 0, 1, 0, 0, 0, 0]
Bob bases:   [0, 0, 1, 0, 0, 0, 0, 1, 1, 0, 1, 1, 1, 0, 0, 1, 0, 0, 0]
Bob bits:    [1, 0, 1, 1, 0, 1, 1, 1, 1, 0, 1, 1, 0, 1, 1, 1, 1, 1, 